In [1]:
from pyspark.sql.session import SparkSession

def get_spark_session(appName="default"):
    spark = (
        SparkSession
        .builder
        .appName(appName)
        .master("local[*]")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

def stop_start(app_name):
   
    global spark 
    
    if 'spark' in globals():
        try:
            spark.stop()
            print("Old Spark session successfully stopped.")
        except Exception as e:
             print(f"No active session to stop. Error: {e}")
    else:
        print("No existing 'spark' variable found.")

    # 3. Return the fresh session
    print("Starting new Spark session...")
    return get_spark_session(app_name)

In [2]:
spark = stop_start("analyzing-query-plan")

No existing 'spark' variable found.
Starting new Spark session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/08 10:00:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

In [18]:
def show(var):
    print(var)

In [41]:
dict = {
    "defaultParallelism": spark.sparkContext.defaultParallelism,
    "shuffle.partions": spark.conf.get("spark.sql.shuffle.partitions"),
    "maxpartitionByte": spark.conf.get("spark.sql.maxPartitionBytes")
}

show(dict)

SparkNoSuchElementException: [SQL_CONF_NOT_FOUND] The SQL config "spark.sql.maxPartitionBytes" cannot be found. Please verify that the config exists. SQLSTATE: 42K0I

In [ ]:
from  pathlib import Path

data_folder = Path("../data") / "optimization-data" 

transaction_data_path = data_folder / "transaction"
customer_data_path = data_folder / "customer" 


if transaction_data_path.exists() and customer_data_path.exists():
    print(True)

True


In [35]:
transaction_df = spark.read.parquet(str(transaction_data_path))
customer_df = spark.read.parquet(str(customer_data_path))

In [38]:
transaction_df.show(5)

+----------+----------+----------+---------------+----------+----+-----+---+-------------+------+-----------+
|   cust_id|start_date|  end_date|         txn_id|      date|year|month|day| expense_type|   amt|       city|
+----------+----------+----------+---------------+----------+----+-----+---+-------------+------+-----------+
|C0YDPQWPBJ|2010-07-01|2018-12-01|TZ5SMKZY9S03OQJ|2018-10-07|2018|   10|  7|Entertainment| 10.42|     boston|
|C0YDPQWPBJ|2010-07-01|2018-12-01|TYIAPPNU066CJ5R|2016-03-27|2016|    3| 27| Motor/Travel| 44.34|   portland|
|C0YDPQWPBJ|2010-07-01|2018-12-01|TETSXIK4BLXHJ6W|2011-04-11|2011|    4| 11|Entertainment|  3.18|    chicago|
|C0YDPQWPBJ|2010-07-01|2018-12-01|TQKL1QFJY3EM8LO|2018-02-22|2018|    2| 22|    Groceries|268.97|los_angeles|
|C0YDPQWPBJ|2010-07-01|2018-12-01|TYL6DFP09PPXMVB|2010-10-16|2010|   10| 16|Entertainment|  2.66|    chicago|
+----------+----------+----------+---------------+----------+----+-----+---+-------------+------+-----------+
only showi

In [39]:
customer_df.show(5)

+----------+-------------+---+------+----------+-----+-----------+
|   cust_id|         name|age|gender|  birthday|  zip|       city|
+----------+-------------+---+------+----------+-----+-----------+
|C007YEYTX9| Aaron Abbott| 34|Female| 7/13/1991|97823|     boston|
|C00B971T1J| Aaron Austin| 37|Female|12/16/2004|30332|    chicago|
|C00WRSJF1Q| Aaron Barnes| 29|Female| 3/11/1977|23451|     denver|
|C01AZWQMF3|Aaron Barrett| 31|  Male|  7/9/1998|46613|los_angeles|
|C01BKUFRHA| Aaron Becker| 54|  Male|11/24/1979|40284|  san_diego|
+----------+-------------+---+------+----------+-----+-----------+
only showing top 5 rows


<img src="../data/pictures/spark_query_plans.jpeg" width="900">